# UN Speeches — Topic Reclassification

Reclassifies `full_dataset_with_bert.csv` (already paragraph-level) with the new 15-topic system.

**⚠️ Runtime**: Use **GPU (T4)** — not TPU. SentenceTransformer uses PyTorch which does not support Colab TPU.
Go to **Runtime → Change runtime type → T4 GPU** before running.

**Upload these files in Cell 2:**
- `full_dataset_with_bert.csv` (from your Downloads folder)
- `topic_definitions.csv` (from `raw_data/` in the project)

**Output files to download at the end:**
- `speeches_paragraphs.csv` → put in `data/`
- `speeches_umap.csv` → put in `streamlit/`
- `topic_labels.csv` → put in `raw_data/`

**Do NOT overwrite** `streamlit/mentioned_countries_agg.csv` — that file is already correct.

In [ ]:
# ── Cell 1: Install dependencies ───────────────────────────────────────────────
!pip install -q sentence-transformers umap-learn scikit-learn pandas

In [ ]:
# ── Cell 2: Upload files ───────────────────────────────────────────────────────
from google.colab import files
print('Upload: full_dataset_with_bert.csv, topic_definitions.csv')
uploaded = files.upload()

In [ ]:
# ── Cell 3: Load and prepare data ─────────────────────────────────────────────
import pandas as pd

df = pd.read_csv('/content/full_dataset_with_bert.csv')
print(f'Loaded: {len(df):,} rows, {df["country"].nunique()} countries, {df["year"].min()}–{df["year"].max()}')

# Rename columns to match expected schema
df = df.rename(columns={
    'score':     'topic_similarity',
    'countries': 'countries_mentioned',
})

# Drop columns from old pipeline we don't need
drop_cols = ['topic_num', 'bert_topic', 'bert_prob', 'ber_topic_words', 'country_dup']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# ── Add continent lookup ───────────────────────────────────────────────────────
ISO_CONTINENT = {
    'AFG':'Asia','ALB':'Europe','DZA':'Africa','AND':'Europe','AGO':'Africa',
    'ATG':'North America','ARG':'South America','ARM':'Asia','AUS':'Oceania',
    'AUT':'Europe','AZE':'Asia','BHS':'North America','BHR':'Asia','BGD':'Asia',
    'BRB':'North America','BLR':'Europe','BEL':'Europe','BLZ':'North America',
    'BEN':'Africa','BTN':'Asia','BOL':'South America','BIH':'Europe','BWA':'Africa',
    'BRA':'South America','BRN':'Asia','BGR':'Europe','BFA':'Africa','BDI':'Africa',
    'CPV':'Africa','KHM':'Asia','CMR':'Africa','CAN':'North America','CAF':'Africa',
    'TCD':'Africa','CHL':'South America','CHN':'Asia','COL':'South America',
    'COM':'Africa','COD':'Africa','COG':'Africa','CRI':'North America','CIV':'Africa',
    'HRV':'Europe','CUB':'North America','CYP':'Europe','CZE':'Europe','CSK':'Europe',
    'CZK':'Europe','DNK':'Europe','DJI':'Africa','DMA':'North America','DOM':'North America',
    'ECU':'South America','EGY':'Africa','SLV':'North America','GNQ':'Africa',
    'ERI':'Africa','EST':'Europe','SWZ':'Africa','ETH':'Africa','FJI':'Oceania',
    'FIN':'Europe','FRA':'Europe','GAB':'Africa','GMB':'Africa','GEO':'Asia',
    'DEU':'Europe','GHA':'Africa','GRC':'Europe','GRD':'North America','GTM':'North America',
    'GIN':'Africa','GNB':'Africa','GUY':'South America','HTI':'North America',
    'HND':'North America','HUN':'Europe','ISL':'Europe','IND':'Asia','IDN':'Asia',
    'IRN':'Asia','IRQ':'Asia','IRL':'Europe','ISR':'Asia','ITA':'Europe','JAM':'North America',
    'JPN':'Asia','JOR':'Asia','KAZ':'Asia','KEN':'Africa','KIR':'Oceania','PRK':'Asia',
    'KOR':'Asia','KWT':'Asia','KGZ':'Asia','LAO':'Asia','LVA':'Europe','LBN':'Asia',
    'LSO':'Africa','LBR':'Africa','LBY':'Africa','LIE':'Europe','LTU':'Europe',
    'LUX':'Europe','MDG':'Africa','MWI':'Africa','MYS':'Asia','MDV':'Asia','MLI':'Africa',
    'MLT':'Europe','MHL':'Oceania','MRT':'Africa','MUS':'Africa','MEX':'North America',
    'FSM':'Oceania','MDA':'Europe','MCO':'Europe','MNG':'Asia','MNE':'Europe','MAR':'Africa',
    'MOZ':'Africa','MMR':'Asia','NAM':'Africa','NRU':'Oceania','NPL':'Asia','NLD':'Europe',
    'NZL':'Oceania','NIC':'North America','NER':'Africa','NGA':'Africa','MKD':'Europe',
    'NOR':'Europe','OMN':'Asia','PAK':'Asia','PLW':'Oceania','PAN':'North America',
    'PNG':'Oceania','PRY':'South America','PER':'South America','PHL':'Asia','POL':'Europe',
    'PRT':'Europe','QAT':'Asia','ROU':'Europe','RUS':'Europe','RWA':'Africa','KNA':'North America',
    'LCA':'North America','VCT':'North America','WSM':'Oceania','SMR':'Europe','STP':'Africa',
    'SAU':'Asia','SEN':'Africa','SRB':'Europe','SLE':'Africa','SGP':'Asia','SVK':'Europe',
    'SVN':'Europe','SLB':'Oceania','SOM':'Africa','ZAF':'Africa','SSD':'Africa','ESP':'Europe',
    'LKA':'Asia','SDN':'Africa','SUR':'South America','SWE':'Europe','CHE':'Europe',
    'SYR':'Asia','TWN':'Asia','TJK':'Asia','TZA':'Africa','THA':'Asia','TLS':'Asia',
    'TGO':'Africa','TON':'Oceania','TTO':'North America','TUN':'Africa','TUR':'Asia',
    'TKM':'Asia','TUV':'Oceania','UGA':'Africa','UKR':'Europe','ARE':'Asia','GBR':'Europe',
    'USA':'North America','URY':'South America','UZB':'Asia','VUT':'Oceania','VEN':'South America',
    'VNM':'Asia','YEM':'Asia','ZMB':'Africa','ZWE':'Africa','YUG':'Europe','SCG':'Europe',
    'DDR':'Europe',
}

df['continent'] = df['iso'].map(ISO_CONTINENT).fillna('Unknown')
df['decade']    = (df['year'] // 10 * 10).astype(str) + 's'
df['year_range'] = df['year'].apply(lambda y: f'{(y//5)*5}-{(y//5)*5+4}')

print('Continents:', df['continent'].value_counts().to_dict())
print('Columns:', df.columns.tolist())

In [ ]:
# ── Cell 4: Topic classification (GPU-accelerated) ────────────────────────────
import gc, numpy as np, torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

topics_df   = pd.read_csv('/content/topic_definitions.csv')
topic_names = topics_df['Name'].str.strip().tolist()
topic_texts = topics_df['Text'].str.strip().tolist()
print('Topics:', topic_names)

model = SentenceTransformer('all-mpnet-base-v2')

print('Encoding topic anchors...')
anchor_embeddings = model.encode(topic_texts, show_progress_bar=False)

paragraphs = df['speeches'].fillna('').tolist()
print(f'Encoding {len(paragraphs):,} paragraphs...')

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

paragraph_embeddings = model.encode(
    paragraphs,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print('Computing similarities...')
similarities = cosine_similarity(paragraph_embeddings, anchor_embeddings)
best_idx     = similarities.argmax(axis=1)
best_score   = similarities.max(axis=1)

df['topic']            = [topic_names[i] for i in best_idx]
df['topic_similarity'] = best_score.round(4)

print('\nNew topic distribution:')
print(df['topic'].value_counts().to_string())
low = (best_score < 0.25).sum()
print(f'\nLow-confidence (< 0.25): {low:,} ({low/len(df)*100:.1f}%)')

In [ ]:
# ── Cell 5: Save speeches_paragraphs.csv ──────────────────────────────────────
OUT_COLS = [
    'iso', 'year', 'speeches', 'Name of Person Speaking', 'order', 'country',
    'speech_length', 'cleaned_speeches', 'preprocessed_speech',
    'topic', 'topic_similarity', 'top_5_words',
    'countries_mentioned', 'countries_recoded',
    'continent', 'decade', 'year_range',
]
# countries_recoded may not exist — fill with empty list if missing
if 'countries_recoded' not in df.columns:
    df['countries_recoded'] = '[]'

out = df[[c for c in OUT_COLS if c in df.columns]]
out.to_csv('/content/speeches_paragraphs.csv', index=False)
print(f'✅ Saved speeches_paragraphs.csv — {len(out):,} rows, {out["country"].nunique()} countries')

In [ ]:
# ── Cell 6: Regenerate speeches_umap.csv ──────────────────────────────────────
import umap as umap_lib

print('Running UMAP (takes a few minutes)...')
reducer     = umap_lib.UMAP(n_components=2, random_state=42)
umap_coords = reducer.fit_transform(paragraph_embeddings)

umap_df  = pd.DataFrame(umap_coords, columns=['umap_1', 'umap_2'])
combined = pd.concat(
    [df[['iso', 'year', 'country', 'continent', 'topic']].reset_index(drop=True), umap_df],
    axis=1
)
speeches_umap = (
    combined
    .groupby(['iso', 'year', 'country', 'continent', 'topic'], dropna=False)
    .agg(umap_1=('umap_1', 'mean'), umap_2=('umap_2', 'mean'), count=('umap_1', 'count'))
    .reset_index()
)
speeches_umap[['umap_1', 'umap_2']] = speeches_umap[['umap_1', 'umap_2']].round(4)
speeches_umap.to_csv('/content/speeches_umap.csv', index=False)
print(f'✅ Saved speeches_umap.csv — {len(speeches_umap):,} rows')

# topic_labels
topic_rows = []
for i, row in topics_df.iterrows():
    name = row['Name'].strip()
    topic_rows.append({
        'topic_id':   i,
        'topic_name': name,
        'count':      int((df['topic'] == name).sum()),
        'top_5_words': str([k.strip() for k in str(row.get('Short_keywords', '')).split(',')][:5]),
    })
pd.DataFrame(topic_rows).to_csv('/content/topic_labels.csv', index=False)
print('✅ Saved topic_labels.csv')

In [ ]:
# ── Cell 7: Download output files ─────────────────────────────────────────────
from google.colab import files

print('Downloading files...')
for fname in ['speeches_paragraphs.csv', 'speeches_umap.csv', 'topic_labels.csv']:
    files.download(f'/content/{fname}')

print()
print('Place downloaded files here:')
print('  speeches_paragraphs.csv  →  data/speeches_paragraphs.csv')
print('  speeches_umap.csv        →  streamlit/speeches_umap.csv')
print('  topic_labels.csv         →  raw_data/topic_labels.csv')
print()
print('⚠️  Do NOT replace streamlit/mentioned_countries_agg.csv')